In [42]:
# Cell 1: Imports and Setup
import sys
import os
import random
import torch
from torch import nn
from torch.optim import AdamW
from torch.utils.data import DataLoader, TensorDataset
from collections import defaultdict
from typing import List
import matplotlib.pyplot as plt

sys.path.append(os.path.abspath("../src"))

from abstractions3d.primitives.shapes import Box3D
from abstractions3d.primitives.visualization import visualize_boxes_3d
from abstractions3d.dsl.core import Shape3D
from abstractions3d.dsl.nodes import Rect3D, Move3D, Union3D, SymTrans3D, SymRef3D
from abstractions3d.dsl.instantiation import instantiate_3d
from abstractions3d.data.blueprint import Table3D, Chair3D, Bench3D

# Set random seeds
random.seed(42)
torch.manual_seed(42)

In [43]:
## Cell 2: Data Generation
def sample_uniform(low: float, high: float) -> float:
    """Samples a random float within a specified range."""
    return random.uniform(low, high)

def generate_dataset(num_samples: int = 50) -> List[Union3D]:
    """Generates a dataset of 3D shapes (tables, chairs, benches)."""
    dataset = []
    for _ in range(num_samples):
        # Generate a Table3D
        dataset.append(Table3D(
            top_length=sample_uniform(3.0, 6.0),
            top_depth=sample_uniform(1.5, 3.0),
            top_thickness=sample_uniform(0.1, 0.3),
            leg_length=sample_uniform(0.2, 0.4),
            leg_depth=sample_uniform(0.2, 0.4),
            leg_height=sample_uniform(1.0, 2.0)
        ))
        
        # Generate a Chair3D
        dataset.append(Chair3D(
            seat_length=sample_uniform(1.0, 2.0),
            seat_depth=sample_uniform(1.0, 2.0),
            seat_thickness=sample_uniform(0.2, 0.5),
            leg_length=sample_uniform(0.1, 0.3),
            leg_depth=sample_uniform(0.1, 0.3),
            leg_height=sample_uniform(0.5, 1.0),
            backrest_height=sample_uniform(1.0, 2.0),
            backrest_thickness=sample_uniform(0.1, 0.3)
        ))
        
        # Generate a Bench3D
        backrest_height = sample_uniform(0.0, 1.5)
        dataset.append(Bench3D(
            seat_length=sample_uniform(2.0, 5.0),
            seat_depth=sample_uniform(0.5, 1.0),
            seat_thickness=sample_uniform(0.2, 0.5),
            leg_length=sample_uniform(0.1, 0.3),
            leg_depth=sample_uniform(0.1, 0.3),
            leg_height=sample_uniform(0.5, 1.0),
            backrest_height=backrest_height,
            # Only give a thickness if there is a backrest
            backrest_thickness=sample_uniform(0.0, 0.3) if backrest_height > 0 else 0.0
        ))
    return dataset


In [44]:
from collections import defaultdict
from typing import Union, List

ShapeType = Union['Shape3D', List['Shape3D']]

def add(structures1: defaultdict, structures2: defaultdict) -> defaultdict:
    for key, value in structures2.items():
        structures1[key] += value
    return structures1


def get_singletons(shapes: ShapeType) -> defaultdict:
    singletons = defaultdict(list)

    if isinstance(shapes, list):
        for shape in shapes:
            singletons = add(singletons, get_singletons(shape))
        return singletons

    shape_type, parameters = shapes.param_tuple()
    singletons[shape_type.__name__].append(parameters)

    # safely handle children
    children = getattr(shapes, 'children', [])
    if children is None:
        children = []

    if isinstance(shapes, (Move3D, SymTrans3D, SymRef3D)) and children:
        singletons = add(singletons, get_singletons(children[0]))
    elif isinstance(shapes, Union3D):
        for child in children:
            singletons = add(singletons, get_singletons(child))

    return singletons


def get_pairs(shapes: ShapeType) -> defaultdict:
    pairs = defaultdict(list)

    if isinstance(shapes, list):
        for shape in shapes:
            pairs = add(pairs, get_pairs(shape))
        return pairs

    children = getattr(shapes, 'children', [])
    if children is None:
        children = []

    if isinstance(shapes, (Move3D, SymTrans3D, SymRef3D)) and children:
        return get_pairs(children[0])
    elif isinstance(shapes, Union3D):
        if len(children) != 2:
            return defaultdict(list)
        type_, (child1, child2) = shapes.param_tuple()
        type1, params1 = child1.param_tuple()
        type2, params2 = child2.param_tuple()
        type_str = f"{type_.__name__}({type1.__name__}, {type2.__name__})"
        current_structures = defaultdict(list)
        current_structures[type_str].append(params1 + params2)
        # include recursively all child pairs
        current_structures = add(current_structures, get_pairs(children[0]))
        current_structures = add(current_structures, get_pairs(children[1]))
        return current_structures

    return defaultdict(list)

In [45]:
# Cell: Generate Dataset and Build Structures
from collections import defaultdict

# 1️⃣ Generate dataset
dataset = generate_dataset(num_samples=50)
print(f"Generated {len(dataset)} shapes.")

# 2️⃣ Extract singleton structures
singleton_structures = get_singletons(dataset)
print(f"Extracted {len(singleton_structures)} singleton types:")
for k, v in singleton_structures.items():
    print(f"  {k}: {len(v)} instances")

# 3️⃣ Extract pair structures
pair_structures = get_pairs(dataset)
print(f"Extracted {len(pair_structures)} pair types:")
for k, v in pair_structures.items():
    print(f"  {k}: {len(v)} instances")

Generated 150 shapes.
Extracted 4 singleton types:
  Union3D: 250 instances
  Move3D: 400 instances
  Rect3D: 400 instances
  SymRef3D: 300 instances
Extracted 3 pair types:
  Union3D(Move3D, SymRef3D): 50 instances
  Union3D(Move3D, Union3D): 100 instances
  Union3D(SymRef3D, Move3D): 100 instances
